[← Course Overview](<00-course-overview.ipynb>)

# Appendix A - SQL Server Reference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/appendix-a-sql-server-reference.ipynb)

The lessons run on SQLite and end on SQL Server, and along the way they point out where the two disagree. This appendix collects those differences in one place, shows how to find your way around a database you've never seen, and gives you a short, runnable tour of the everyday T-SQL tools the lessons didn't need: paging, dates, text, and missing values. It's a reference to come back to, not a lesson to work through in order.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

## 0. Set up

Every cell in this appendix runs on SQL Server, so run both setup cells: the first loads the helper, and the second gets AdventureWorks ready (in Colab it installs SQL Server, which takes a few minutes the first time; on your own machine it checks the connection).

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


## 1. SQL Server in one page

**SQL Server** is Microsoft's database engine, and **T-SQL** (Transact-SQL) is its dialect of SQL. The editions you're likely to meet:

| Edition | What it's for |
| --- | --- |
| **Developer** | Every feature, free, licensed for learning, development, and testing, not for running a real business. What this course uses. |
| **Express** | Free for production use, with limits on database size and memory. |
| **Standard** and **Enterprise** | Paid editions that companies run their systems on. |
| **Azure SQL Database** | SQL Server's engine as a cloud service, with nothing to install. |

The tools you'll hear about:

- **SQL Server Management Studio (SSMS)** is the classic Windows application for connecting to SQL Server, writing queries, and managing databases, including restoring the AdventureWorks backup.
- **VS Code with the mssql extension** writes and runs `.sql` files against SQL Server on any operating system.
- **Azure Data Studio**, which earlier versions of this repo used, was retired by Microsoft in February 2026.
- **This notebook's `%%tsql` cells** connect from Python, using `pyodbc` with your Windows login or `pymssql` with a username and password.

## 2. Finding your way around a database

On a database you've never seen, the first question is the one from `01`: what tables are there, and what does one row of each represent? SQL Server describes itself through **`INFORMATION_SCHEMA`** views, which are standard SQL, so the same queries work on PostgreSQL and MySQL too. The tables in one schema:

In [3]:
%%tsql
SELECT
    TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'Sales'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY
    TABLE_NAME;

TABLE_NAME
CountryRegionCurrency
CreditCard
Currency
CurrencyRate
Customer
PersonCreditCard
SalesOrderDetail
SalesOrderHeader
SalesOrderHeaderSalesReason
SalesPerson


The columns of one table, with their types and whether they can be `NULL`:

In [4]:
%%tsql
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH,
    IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'Sales'
  AND TABLE_NAME = 'SalesOrderHeader'
ORDER BY
    ORDINAL_POSITION;

COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
SalesOrderID,int,NULL,NO
RevisionNumber,tinyint,NULL,NO
OrderDate,datetime,NULL,NO
DueDate,datetime,NULL,NO
ShipDate,datetime,NULL,YES
Status,tinyint,NULL,NO
OnlineOrderFlag,bit,NULL,NO
SalesOrderNumber,nvarchar,25,NO
PurchaseOrderNumber,nvarchar,25,YES
AccountNumber,nvarchar,15,YES


`IS_NULLABLE` is worth reading before you filter or average a column: `SalesPersonID` says `YES`, which is the 27,659 online orders `02` found missing from both sides of a filter. And the foreign keys, which say how the tables connect (the `08b` stretch exercise used the same system view):

In [5]:
%%tsql
SELECT
    fk.name AS foreign_key,
    OBJECT_SCHEMA_NAME(fk.parent_object_id) + '.' + OBJECT_NAME(fk.parent_object_id) AS child_table,
    OBJECT_SCHEMA_NAME(fk.referenced_object_id) + '.' + OBJECT_NAME(fk.referenced_object_id) AS parent_table
FROM sys.foreign_keys AS fk
WHERE OBJECT_SCHEMA_NAME(fk.parent_object_id) = 'Sales'
  AND OBJECT_NAME(fk.parent_object_id) = 'SalesOrderHeader'
ORDER BY
    fk.name;

foreign_key,child_table,parent_table
FK_SalesOrderHeader_Address_BillToAddressID,Sales.SalesOrderHeader,Person.Address
FK_SalesOrderHeader_Address_ShipToAddressID,Sales.SalesOrderHeader,Person.Address
FK_SalesOrderHeader_CreditCard_CreditCardID,Sales.SalesOrderHeader,Sales.CreditCard
FK_SalesOrderHeader_CurrencyRate_CurrencyRateID,Sales.SalesOrderHeader,Sales.CurrencyRate
FK_SalesOrderHeader_Customer_CustomerID,Sales.SalesOrderHeader,Sales.Customer
FK_SalesOrderHeader_SalesPerson_SalesPersonID,Sales.SalesOrderHeader,Sales.SalesPerson
FK_SalesOrderHeader_SalesTerritory_TerritoryID,Sales.SalesOrderHeader,Sales.SalesTerritory
FK_SalesOrderHeader_ShipMethod_ShipMethodID,Sales.SalesOrderHeader,Purchasing.ShipMethod


## 3. Data types you'll meet

| Type | Holds | Notes |
| --- | --- | --- |
| `INT`, `BIGINT`, `SMALLINT`, `TINYINT` | Whole numbers | Dividing two of them gives a whole number (`03`, `09`). |
| `DECIMAL(p, s)` / `NUMERIC(p, s)` | Exact decimals, `p` digits with `s` after the point | Use for amounts that must add up exactly. |
| `MONEY` | Currency, four decimal places | AdventureWorks uses it for `TotalDue`. |
| `FLOAT`, `REAL` | Approximate decimals | Fast, but 0.1 + 0.2 isn't exactly 0.3; avoid for money. |
| `VARCHAR(n)` | Text up to `n` characters | |
| `NVARCHAR(n)` | Unicode text up to `n` characters | Needed for names in any script beyond plain English letters; write literals as `N'Ọlá'`. |
| `DATE`, `DATETIME2`, `DATETIME` | A date, or a date and time | Prefer `DATETIME2` for new tables; AdventureWorks predates it and uses `DATETIME`. |
| `BIT` | 0, 1, or `NULL` | SQL Server's yes/no type, like `OnlineOrderFlag` in `09`. |

## 4. T-SQL and SQLite side by side

Every difference the lessons ran into, in one table, with the notebook that shows it:

| Task | SQLite | SQL Server | Where it comes up |
| --- | --- | --- | --- |
| First *n* rows | `... LIMIT 5` | `SELECT TOP 5 ...` | `02` onwards |
| Skip rows, then take some | `LIMIT 5 OFFSET 10` | `ORDER BY ... OFFSET 10 ROWS FETCH NEXT 5 ROWS ONLY` | Section 5 below |
| Alias in `WHERE` | Accepted (non-standard) | Error: Invalid column name | `02` |
| Text comparison | Case-sensitive by default | Depends on the collation; AdventureWorks is case-insensitive | `02` |
| Missing values in a sort | Smallest | Smallest | `02` |
| `NULLS LAST` | Supported | Not in SQL Server 2022; sort on a `CASE` first | `02` |
| `AVG` of whole numbers | Always a decimal | A whole number, truncated | `03` |
| Whole number / whole number | Whole number | Whole number | `09` |
| Bare column in a `GROUP BY` | Accepted, picks a row | Error | `03` |
| `FULL OUTER JOIN` | From SQLite 3.39.0 | Supported | `04` |
| Subquery returning several rows as one value | Uses the first row silently | Error | `05` |
| Derived table without an alias | Accepted | Error | `05` |
| Recursive CTE keyword | `WITH RECURSIVE` (optional in SQLite) | `WITH` only; `RECURSIVE` is an error | `05` |
| `AS MATERIALIZED` | From SQLite 3.35.0 | Not supported | `05` |
| `RANGE` frame with a numeric offset | Supported | Only `UNBOUNDED` and `CURRENT ROW` | `07` |
| Start a transaction | `BEGIN;` | `BEGIN TRANSACTION;` | `08a` |
| Rows changed by the last statement | `SELECT changes();` | `SELECT @@ROWCOUNT;` | `08a` |
| Temporary table | `CREATE TEMP TABLE` | A name starting with `#` | `08a` |
| Copy rows into a new table | `CREATE TABLE new AS SELECT ...` | `SELECT ... INTO new FROM ...` | `08a` |
| Column types | Per value, unless the table is `STRICT` | Fixed per column | `08b` |
| Foreign keys | Off until `PRAGMA foreign_keys = ON;` | Always enforced | `08b` |
| `UNIQUE` column with `NULL`s | Any number of `NULL`s | Only one `NULL` | `08b` |
| Add a column | `ADD COLUMN` | `ADD` | `08b` |
| Change a column's type, add a constraint | Rebuild the table | `ALTER COLUMN`, `ADD CONSTRAINT` | `08b` |
| Join text | `a || b` | `a + b`, or `CONCAT(a, b)` | Section 5 below |
| Today's date | `date('now')` | `CAST(GETDATE() AS DATE)` | |
| Separate batches in one script | Not needed | `GO` on its own line | `08b` |

## 5. Everyday T-SQL tools

### Paging through results

`TOP` takes the first rows; to skip some first, as a website showing "page 3" of a list would, SQL Server uses `OFFSET` and `FETCH`, which require an `ORDER BY`:

In [6]:
%%tsql
SELECT
    SalesOrderID,
    OrderDate,
    TotalDue
FROM Sales.SalesOrderHeader
ORDER BY
    SalesOrderID
OFFSET 10 ROWS
FETCH NEXT 3 ROWS ONLY;

SalesOrderID,OrderDate,TotalDue
43669,2011-05-31 00:00:00,807.2585
43670,2011-05-31 00:00:00,6893.2549
43671,2011-05-31 00:00:00,9153.6054


Rows 11 to 13. Without a unique `ORDER BY`, the same "page" can show different rows each time, the tie problem from `07` in another form.

### Dates

In [7]:
%%tsql
SELECT TOP 3
    OrderDate,
    DueDate,
    YEAR(OrderDate) AS order_year,
    DATEDIFF(day, OrderDate, DueDate) AS days_until_due,
    DATEADD(month, 1, OrderDate) AS one_month_later,
    EOMONTH(OrderDate) AS end_of_month
FROM Sales.SalesOrderHeader
ORDER BY
    SalesOrderID;

OrderDate,DueDate,order_year,days_until_due,one_month_later,end_of_month
2011-05-31 00:00:00,2011-06-12 00:00:00,2011,12,2011-06-30 00:00:00,2011-05-31
2011-05-31 00:00:00,2011-06-12 00:00:00,2011,12,2011-06-30 00:00:00,2011-05-31
2011-05-31 00:00:00,2011-06-12 00:00:00,2011,12,2011-06-30 00:00:00,2011-05-31


One month after 31 May is 30 June, because `DATEADD` stops at the end of a shorter month rather than spilling into July. `DATEDIFF` counts boundaries crossed, not full periods, so `DATEDIFF(year, '2013-12-31', '2014-01-01')` is 1, even though the dates are a day apart. When filtering a date range, `OrderDate >= '2013-01-01' AND OrderDate < '2014-01-01'` is safer than `BETWEEN`, because it can't miss a time later on the last day.

### Text, and a NULL trap

Building a full name from parts. `MiddleName` is `NULL` for many people, and the three ways of joining text treat that differently:

In [8]:
%%tsql
SELECT TOP 3
    FirstName,
    MiddleName,
    LastName,
    FirstName + ' ' + MiddleName + ' ' + LastName AS with_plus,
    CONCAT(FirstName, ' ', MiddleName, ' ', LastName) AS with_concat,
    CONCAT_WS(' ', FirstName, MiddleName, LastName) AS with_concat_ws
FROM Person.Person
WHERE MiddleName IS NULL
ORDER BY
    BusinessEntityID;

FirstName,MiddleName,LastName,with_plus,with_concat,with_concat_ws
Roberto,NULL,Tamburello,NULL,Roberto Tamburello,Roberto Tamburello
Rob,NULL,Walters,NULL,Rob Walters,Rob Walters
Michael,NULL,Raheem,NULL,Michael Raheem,Michael Raheem


With `+`, one `NULL` part makes the whole name `NULL`, the same rule as `NULL` in arithmetic from `06`. `CONCAT` treats a `NULL` as empty text, which keeps the name but leaves a double space. `CONCAT_WS` ("with separator") skips the `NULL` and its separator, and is usually what you want.

### Replacing missing values

`ISNULL` is SQL Server's own version of the standard `COALESCE`. `COALESCE` takes any number of arguments and returns the first that isn't `NULL`, and it works on every engine, so prefer it:

In [9]:
%%tsql
SELECT TOP 3
    SalesOrderID,
    SalesPersonID,
    COALESCE(SalesPersonID, 0) AS salesperson_or_zero,
    ISNULL(CAST(SalesPersonID AS VARCHAR(10)), 'online') AS salesperson_label
FROM Sales.SalesOrderHeader
ORDER BY
    SalesOrderID DESC;

SalesOrderID,SalesPersonID,salesperson_or_zero,salesperson_label
75123,NULL,0,online
75122,NULL,0,online
75121,NULL,0,online


### Joining a group's values into one string

`STRING_AGG` is an aggregate, like `SUM`, that joins text instead of adding numbers. Here are the salespeople in each territory:

In [10]:
%%tsql
SELECT
    t.Name AS territory,
    COUNT(*) AS salespeople,
    STRING_AGG(CAST(p.BusinessEntityID AS VARCHAR(10)), ', ') AS salesperson_ids
FROM Sales.SalesTerritory AS t
INNER JOIN Sales.SalesPerson AS p
    ON p.TerritoryID = t.TerritoryID
GROUP BY
    t.Name
ORDER BY
    t.Name;

territory,salespeople,salesperson_ids
Australia,1,286
Canada,2,"278, 282"
Central,1,277
France,1,290
Germany,1,288
Northeast,1,275
Northwest,3,"280, 283, 284"
Southeast,1,279
Southwest,2,"276, 281"
United Kingdom,1,289


## 6. Before you DROP anything

`DROP TABLE` removes a table and all its rows, and `DROP DATABASE` removes a whole database; neither asks for confirmation, and outside a transaction neither can be undone. Before running one on anything that matters:

- Take a backup, and check it restores. Microsoft's [Back up and restore SQL Server databases](https://learn.microsoft.com/en-us/sql/relational-databases/backup-restore/back-up-and-restore-of-sql-server-databases) explains the options.
- Check what depends on the object; the foreign-key query in section 2 is a start.
- Practise on a copy, the way `08a` and `08b` used `#` tables.

## Go deeper

- **[Transact-SQL reference](https://learn.microsoft.com/en-us/sql/t-sql/language-reference)**, Microsoft Learn. The full T-SQL reference, organized by statement and function.
- **[Data types](https://learn.microsoft.com/en-us/sql/t-sql/data-types/data-types-transact-sql)**, Microsoft Learn. Every SQL Server type, with its range and storage size.
- **[Date and time functions](https://learn.microsoft.com/en-us/sql/t-sql/functions/date-and-time-data-types-and-functions-transact-sql)** and **[String functions](https://learn.microsoft.com/en-us/sql/t-sql/functions/string-functions-transact-sql)**, Microsoft Learn. The complete lists behind section 5.
- **[Modern SQL](https://modern-sql.com/)**, Markus Winand. Which standard SQL features each engine supports, with notes on where they disagree, the bigger version of section 4.

---

[← Course Overview](<00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).